In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ultranest

In [26]:
data = []
with open("data.txt") as f:
    lines = f.readlines()[36:]
    for line in lines:
        if line[61] == "#":
            continue
        el = line[20:22].strip()
        N = int(line[6:9].strip())
        Z = int(line[11:14].strip())
        A = int(line[16:19].strip())
        binding_energy = float(line[56:67].strip()) * A / 1000
        binding_energy_std = float(line[68:77].strip()) * A / 1000
        data.append([el, N, Z, A, binding_energy, binding_energy_std])
df = pd.DataFrame(data, columns=["Element", "N", "Z", "A", "Binding Energy (MeV)", "Binding Energy std (MeV)"])

In [30]:
df

,Element,N,Z,A,Binding Energy (MeV),Binding Energy std (MeV)
0,n,1,0,1,0.000000,0.000000e+00
1,H,0,1,1,0.000000,0.000000e+00
2,H,1,1,2,2.224566,4.000000e-07
3,H,2,1,3,8.481796,9.000000e-07
4,He,1,2,3,7.718041,3.000000e-07
...,...,...,...,...,...,...
2545,Hs,157,108,265,1933.505561,2.395600e-02
2546,Hs,158,108,266,1941.337453,2.710540e-02
2547,Mt,157,109,266,1934.022107,9.647820e-02
2548,Ds,159,110,269,1950.291722,3.139230e-02


In [43]:
def model_simple(N, Z, A, a_v, a_s, a_c, a_a):
    return a_v*A - a_s*A**(2/3)-a_c*Z*(Z-1)/A**(1/3)-a_a*(N-Z)**2/A

def prior_transform_simple(cube):
    params = np.zeros_like(cube)

    a_v_min = 0.1
    a_v_max = 20.0
    params[0] = 10**(cube[0] * (np.log10(a_v_max) - np.log10(a_v_min)) + np.log10(a_v_min))

    a_s_min = 0.1
    a_s_max = 20.0
    params[1] = 10**(cube[1] * (np.log10(a_s_max) - np.log10(a_s_min)) + np.log10(a_s_min))

    a_c_min = 0.1
    a_c_max = 20.0
    params[2] = 10**(cube[2] * (np.log10(a_c_max) - np.log10(a_c_min)) + np.log10(a_c_min))

    a_a_min = 0.1
    a_a_max = 20.0
    params[3] = 10**(cube[3] * (np.log10(a_a_max) - np.log10(a_a_min)) + np.log10(a_a_min))

    return params

def likelihood_simple(params):
    model = model_simple(df["N"], df["Z"], df["A"].values, *params)

    residual = (df["Binding Energy (MeV)"] - model) / (df["Binding Energy std (MeV)"] + 1e-9)

    log_norm = np.sum(np.log(2 * np.pi * (df["Binding Energy std (MeV)"] + 1e-9)**2))

    return -0.5 * (np.sum(residual**2) + log_norm)

params_simple = ["a_v", "a_s", "a_c", "a_a"]

In [ ]:
sampler_simple = ultranest.ReactiveNestedSampler(
    params_simple, likelihood_simple, prior_transform_simple
)
result_simple = sampler_simple.run()
sampler_simple.print_results()

[ultranest] Sampling 400 live points from prior ...


/usr/lib/python3.14/site-packages/ultranest/integrator.py:1903: UserWarning: Sampling from region seems inefficient (0/40 accepted in iteration 2500). To improve efficiency, modify the transformation so that the current live points are ellipsoidal, or use a stepsampler, or set frac_remain to a lower number (e.g., 0.5) to terminate earlier.
  u, v, logl, nc, quality = self._refill_samples(Lmin, ndraw, nit)


[ultranest] Explored until L=-2e+14  e+14 [-2.391e+14..-2.391e+14] | it/evals=16380/71883 eff=22.9145% N=400 
[ultranest] Likelihood function evaluations: 71883
[ultranest]   logZ = -2.391e+14 +- 8.51e+07
[ultranest] Effective samples strategy satisfied (ESS = 1.0, need >400)
[ultranest] Posterior uncertainty strategy wants to improve: -1054237610812081831936.00..-239114620999984.56 (KL: 47863475.12+-233002978.29 nat, need <0.50 nat)
[ultranest] Evidency uncertainty strategy wants 398 minimum live points (dlogz from 47863475.23 to 280866452.84, need <0.5)
[ultranest]   logZ error budget: single: 0.34 bs:85098665.94 tail:0.69 total:85098665.94 required:<0.50


/usr/lib/python3.14/site-packages/ultranest/integrator.py:1686: RuntimeWarning: invalid value encountered in divide
  p /= p.sum(axis=0).reshape((1, -1))


[ultranest] Widening from 1 to 800 live points before L=-1e+21...
[ultranest] parent value is -inf, so widening roots
[ultranest] Widening roots to 800 live points (have 400 already) ...
[ultranest] Sampling 400 live points from prior ...
[ultranest] Exploring (in particular: L=-inf..-239114620999984.56) ...
[ultranest] Explored until L=-2e+14  e+14 [-inf..-2.391e+14] | it/evals=32586/142439 eff=23.0985% N=800 
[ultranest] Likelihood function evaluations: 142439
[ultranest]   logZ = -2.391e+14 +- 2.829e+07
[ultranest] Effective samples strategy satisfied (ESS = 1.0, need >400)
[ultranest] Posterior uncertainty strategy wants to improve: -1260585022626924331008.00..-239114620999984.56 (KL: 18487474.09+-48634657.00 nat, need <0.50 nat)


/usr/lib/python3.14/site-packages/ultranest/integrator.py:1686: RuntimeWarning: invalid value encountered in divide
  p /= p.sum(axis=0).reshape((1, -1))


[ultranest] Evidency uncertainty strategy wants 798 minimum live points (dlogz from 18487474.32 to 67122130.53, need <0.5)
[ultranest]   logZ error budget: single: 0.24 bs:28285906.96 tail:0.69 total:28285906.96 required:<0.50
[ultranest] Widening roots to 798 live points (have 800 already) ...
[ultranest] Exploring (in particular: L=-1260585022626924331008.00..-239114620999984.56) ...
[ultranest] Explored until L=-2e+14  e+14 [-1.261e+21..-2.391e+14] | it/evals=32587/142444 eff=0.0000% N=800 
[ultranest] Likelihood function evaluations: 142444
[ultranest]   logZ = -2.391e+14 +- 2.294e+07
[ultranest] Effective samples strategy satisfied (ESS = 1.0, need >400)
[ultranest] Posterior uncertainty strategy wants to improve: -1260585022626924331008.00..-239114620999984.56 (KL: 10244641.72+-56877489.38 nat, need <0.50 nat)
[ultranest] Evidency uncertainty strategy wants 798 minimum live points (dlogz from 10244641.96 to 67122130.91, need <0.5)
[ultranest]   logZ error budget: single: 0.24 bs:

/usr/lib/python3.14/site-packages/ultranest/integrator.py:1686: RuntimeWarning: invalid value encountered in divide
  p /= p.sum(axis=0).reshape((1, -1))


[ultranest] Widening from 1 to 1600 live points before L=-1e+21...
[ultranest] parent value is -inf, so widening roots
[ultranest] Widening roots to 1600 live points (have 800 already) ...
[ultranest] Sampling 800 live points from prior ...
[ultranest] Exploring (in particular: L=-inf..-239114620999984.56) ...


In [ ]:
A_range = np.arange(np.min(df["A"]), np.max(df["A"]))

b = np.zeros_like(A)

for i, A in enumerate(A_range):
    for N in range(0, A + 1):
        Z = A - N
        en = model_simple(N, Z, A, *result_simple["posterior"]["mean"])
        b[i] = np.max(b[i], en)

plt.plot(A_range, b, linestyle='', marker='.', label="Modello semplice")

plt.errorbar(df["A"], df["Binding Energy (MeV)"] / df["A"], df["Binding Energy std (MeV)"] / df["A"], label="Dati")

plt.legend()
plt.plot()

In [ ]:
def model_pairing(N, Z, A, a_v, a_s, a_c, a_a, a_p):
    delta_0 = a_p * A**(-1/2)
    # (A % 2 == 1) * ((N % 2 == 0) * 2 - 1)
    b = a_v * A \
        - a_s * A**(2/3) \
        - a_c * Z * (Z - 1) / A**(1/3) \
        - a_a * (N - Z)**2 / A \
        + np.where(A % 2 == 1, 0, np.where(N % 2 == 0, 1, -1)) * delta_0
    return b

def prior_transform_pairing(cube):
    params = np.zeros_like(cube)

    a_v_min = 0.1
    a_v_max = 20.0
    params[0] = 10**(cube[0] * (np.log10(a_v_max) - np.log10(a_v_min)) + np.log10(a_v_min))

    a_s_min = 0.1
    a_s_max = 20.0
    params[1] = 10**(cube[1] * (np.log10(a_s_max) - np.log10(a_s_min)) + np.log10(a_s_min))

    a_c_min = 0.1
    a_c_max = 20.0
    params[2] = 10**(cube[2] * (np.log10(a_c_max) - np.log10(a_c_min)) + np.log10(a_c_min))

    a_a_min = 0.1
    a_a_max = 20.0
    params[3] = 10**(cube[3] * (np.log10(a_a_max) - np.log10(a_a_min)) + np.log10(a_a_min))

    a_p_min = 5
    a_p_max = 40
    params[4] = 10**(cube[4] * (np.log10(a_p_max) - np.log10(a_p_min)) + np.log10(a_p_min))

    return params

def likelihood_pairing(params):
    model = model_pairing(df["N"], df["Z"], df["A"].values, *params)

    residual = (df["Binding Energy (MeV)"] - model) / (df["Binding Energy std (MeV)"] + 1e-9)

    log_norm = np.sum(np.log(2 * np.pi * (df["Binding Energy std (MeV)"] + 1e-9)**2))

    return -0.5 * (np.sum(residual**2) + log_norm)

params_pairing = ["a_v", "a_s", "a_c", "a_a", "a_p"]

In [ ]:
sampler_pairing = ultranest.ReactiveNestedSampler(
    params_pairing, likelihood_pairing, prior_transform_pairing
)
result_pairing = sampler_pairing.run()
sampler_pairing.print_results()

[ultranest] Sampling 400 live points from prior ...


/usr/lib/python3.14/site-packages/ultranest/integrator.py:1903: UserWarning: Sampling from region seems inefficient (0/40 accepted in iteration 2500). To improve efficiency, modify the transformation so that the current live points are ellipsoidal, or use a stepsampler, or set frac_remain to a lower number (e.g., 0.5) to terminate earlier.
  u, v, logl, nc, quality = self._refill_samples(Lmin, ndraw, nit)


[ultranest] Explored until L=-2e+14  e+14 [-2.391e+14..-2.391e+14] | it/evals=16380/71883 eff=22.9145% N=400 
[ultranest] Likelihood function evaluations: 71883
[ultranest]   logZ = -2.391e+14 +- 8.51e+07
[ultranest] Effective samples strategy satisfied (ESS = 1.0, need >400)
[ultranest] Posterior uncertainty strategy wants to improve: -1054237610812081831936.00..-239114620999984.56 (KL: 47863475.12+-233002978.29 nat, need <0.50 nat)
[ultranest] Evidency uncertainty strategy wants 398 minimum live points (dlogz from 47863475.23 to 280866452.84, need <0.5)
[ultranest]   logZ error budget: single: 0.34 bs:85098665.94 tail:0.69 total:85098665.94 required:<0.50


/usr/lib/python3.14/site-packages/ultranest/integrator.py:1686: RuntimeWarning: invalid value encountered in divide
  p /= p.sum(axis=0).reshape((1, -1))


[ultranest] Widening from 1 to 800 live points before L=-1e+21...
[ultranest] parent value is -inf, so widening roots
[ultranest] Widening roots to 800 live points (have 400 already) ...
[ultranest] Sampling 400 live points from prior ...
[ultranest] Exploring (in particular: L=-inf..-239114620999984.56) ...
[ultranest] Explored until L=-2e+14  e+14 [-inf..-2.391e+14] | it/evals=32586/142439 eff=23.0985% N=800 
[ultranest] Likelihood function evaluations: 142439
[ultranest]   logZ = -2.391e+14 +- 2.829e+07
[ultranest] Effective samples strategy satisfied (ESS = 1.0, need >400)
[ultranest] Posterior uncertainty strategy wants to improve: -1260585022626924331008.00..-239114620999984.56 (KL: 18487474.09+-48634657.00 nat, need <0.50 nat)
